# Preferences and distillation

Pairwise preferences, rewards, and teacher distributions.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

SFT uses target answers. DPO uses chosen/rejected pairs and the policy-to-reference ratio. Distillation transfers a teacher distribution through KL. These are different learning signals.

We compute DPO from sequence log-probabilities and distillation from logits. Text training needs log-prob sums over the supervised tokens, matching tokenizers for direct logit distillation, and separate evaluation. Available Studio modes depend on the model family.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
chosen = torch.tensor([-2.0, -1.4], requires_grad=True)
rejected = torch.tensor([-3.0, -2.0], requires_grad=True)
ref_chosen = torch.tensor([-2.5, -1.7]); ref_rejected = torch.tensor([-2.8, -2.1])
beta = 0.1
dpo_loss = -F.logsigmoid(beta * ((chosen - rejected) - (ref_chosen - ref_rejected))).mean()
dpo_loss.backward()
print('DPO:', dpo_loss.item(), 'chosen gradient:', chosen.grad)
teacher = torch.tensor([[3., 1., -1.]])
student = torch.tensor([[0., 0., 0.]], requires_grad=True)
temperature = 2.
kd = F.kl_div((student / temperature).log_softmax(-1), (teacher / temperature).softmax(-1), reduction='batchmean') * temperature**2
kd.backward(); print('distillation KL:', kd.item(), 'student gradient:', student.grad)

## Try it yourself

Swap chosen and rejected and inspect the gradient sign. Change temperature: which differences in the teacher distribution become more visible?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [DPO](https://arxiv.org/abs/2305.18290)
- [Distillation](https://arxiv.org/abs/1503.02531)